# Derivation: The EWLS Recursion

In this derivation, we obtain the exponentially weighted least squares (EWLS) recursion used by Session 3's online engine to track the SIM parameters $(\alpha_{i}, \beta_{i}, \sigma_{\varepsilon,i})$ as new market days arrive. We derive the running sufficient statistics $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ from the weighted normal equations, recover the parameter estimate and residual scale, and show that the prior weight $W_{0}$ and half-life $h$ are the only knobs that move the estimator along the speed-stability frontier.

> __Learning Objectives:__
>
> By the end of this notebook, we will be able to:
> * __Weighted normal equations:__ We will derive the EWLS first-order conditions from the exponentially weighted squared-residual loss. We will see why three running moments, a $2 \times 2$ matrix, a 2-vector, and a scalar, are sufficient for the recursion.
> * __Mechanical recursion:__ We will show how each running moment updates as a single decayed-plus-rank-one step when time advances. We will read off the full EWLS update from the loss without storing raw history.
> * __Residual scale and prior seeding:__ We will recover the residual standard deviation from the same sufficient statistics and seed the recursion from a calibrated prior with weight $W_{0}$. We will be able to predict how the half-life $h$ and prior weight $W_{0}$ trade speed against stability.

___


## SIM in Vector Form and the Weighted Loss

We want to track the SIM parameters $(\alpha_{i}, \beta_{i})$ for asset $i$ as new market days arrive, without storing every past observation. Fix asset $i$ and at each step $s = 1, 2, \ldots, t$ observe the asset continuously compounded growth rate (CCGR) $g_{i,s}$ and the market CCGR $g_{\mathrm{mkt},s}$. Stack the regressors with an intercept into the augmented vector $\mathbf{x}_{s} = [1,\; g_{\mathrm{mkt},s}]^{\top} \in \mathbb{R}^{2}$, write the SIM parameters as $\boldsymbol{\theta}_{i} = [\alpha_{i},\; \beta_{i}]^{\top}$, and write the target as the scalar $y_{s} = g_{i,s}$. The single-asset SIM regression collapses to:
$$
y_{s} = \mathbf{x}_{s}^{\top}\boldsymbol{\theta}_{i} + \varepsilon_{i,s}
$$

Old observations should fade as new ones arrive. Pick a half-life $h$ in trading days and define the decay factor $\delta = 2^{-1/h} \in (0, 1)$. The weight at time $t$ on the observation made at step $s \le t$ is given by:
$$
w_{s,t} = \delta^{\,t-s}
$$
An observation made $h$ steps before $t$ carries half the weight of today's observation since $\delta^{h} = 1/2$. The exponentially weighted least squares estimate of $\boldsymbol{\theta}_{i}$ at time $t$ is the minimizer of the corresponding weighted squared residual:
$$
L_{t}(\boldsymbol{\theta}) = \sum_{s=1}^{t} w_{s,t}\,(y_{s} - \mathbf{x}_{s}^{\top}\boldsymbol{\theta})^{2}
$$

This is a quadratic objective in $\boldsymbol{\theta}$, so the minimizer is unique whenever the weighted regressor moments have full rank, which holds as soon as $g_{\mathrm{mkt}}$ has shown any variation in the weighted window.

___


## Weighted Normal Equations and Sufficient Statistics

Setting $\nabla_{\boldsymbol{\theta}} L_{t} = \mathbf{0}$ produces the weighted normal equations $\mathbf{A}_{t}\,\boldsymbol{\theta} = \mathbf{b}_{t}$, where the two weighted moments needed for the optimum are $\mathbf{A}_{t} \in \mathbb{R}^{2 \times 2}$ and $\mathbf{b}_{t} \in \mathbb{R}^{2}$, and we keep one extra weighted scalar moment $c_{t} \in \mathbb{R}$ that returns later when we estimate the residual variance:
$$
\mathbf{A}_{t} = \sum_{s=1}^{t} w_{s,t}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top}, \qquad \mathbf{b}_{t} = \sum_{s=1}^{t} w_{s,t}\,\mathbf{x}_{s} y_{s}, \qquad c_{t} = \sum_{s=1}^{t} w_{s,t}\,y_{s}^{2}
$$

Everything EWLS needs is encoded in these three running quantities. The $(1,1)$ entry of $\mathbf{A}_{t}$ is the effective sample weight $S_{w,t} = \sum_{s} w_{s,t}$ since $\mathbf{x}_{s}\mathbf{x}_{s}^{\top}$ has a $1$ in the top-left; the off-diagonal carries $\sum_{s} w_{s,t}\, g_{\mathrm{mkt},s}$, and the $(2,2)$ entry carries $\sum_{s} w_{s,t}\, g_{\mathrm{mkt},s}^{2}$. The vector $\mathbf{b}_{t}$ stacks $\sum_{s} w_{s,t}\, y_{s}$ and $\sum_{s} w_{s,t}\, g_{\mathrm{mkt},s}\, y_{s}$, and $c_{t}$ is the weighted target second moment. Together $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ are the *sufficient statistics* of the weighted regression: any quantity EWLS reports at time $t$ is a deterministic function of these three running totals.

___


## The Mechanical Recursion

Each moment is a weighted sum, and when time advances from $t-1$ to $t$ every old weight $w_{s,t-1} = \delta^{(t-1)-s}$ becomes $\delta\cdot w_{s,t-1} = \delta^{t-s}$, while the new observation $(\mathbf{x}_{t}, y_{t})$ enters with weight $w_{t,t} = \delta^{0} = 1$. Splitting each moment into its history and its today contribution gives:
$$
\mathbf{A}_{t} = \sum_{s=1}^{t-1} \delta^{t-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top} = \delta \sum_{s=1}^{t-1} \delta^{(t-1)-s}\,\mathbf{x}_{s}\mathbf{x}_{s}^{\top} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top} = \delta\,\mathbf{A}_{t-1} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}
$$
and identically for $\mathbf{b}_{t}$ and $c_{t}$. The recursion is therefore mechanical:
$$
\boxed{\;\mathbf{A}_{t} \leftarrow \delta\,\mathbf{A}_{t-1} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}, \qquad \mathbf{b}_{t} \leftarrow \delta\,\mathbf{b}_{t-1} + \mathbf{x}_{t} y_{t}, \qquad c_{t} \leftarrow \delta\,c_{t-1} + y_{t}^{2}\quad\blacksquare\;}
$$

A $2 \times 2$ rank-one matrix add, a 2-vector add, and a scalar add: the entire weighted past is compressed into $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ with no raw history retained. The recursion uses $O(1)$ memory and $O(1)$ work per step.

___


## Closed-Form Parameter Estimate

Solving the weighted normal equations $\mathbf{A}_{t}\,\boldsymbol{\theta} = \mathbf{b}_{t}$ gives the running SIM estimate:
$$
\boxed{\;\hat{\boldsymbol{\theta}}_{i,t} = \begin{bmatrix} \hat{\alpha}_{i,t} \\ \hat{\beta}_{i,t} \end{bmatrix} = \mathbf{A}_{t}^{-1}\,\mathbf{b}_{t}\quad\blacksquare\;}
$$

Because $\mathbf{A}_{t}$ is $2 \times 2$, this is a single closed-form solve per day rather than a full matrix inversion. Writing $\mathbf{A}_{t} = \begin{bmatrix} S_{w} & S_{wm} \\ S_{wm} & S_{wmm} \end{bmatrix}$ and $\mathbf{b}_{t} = \begin{bmatrix} S_{wy} \\ S_{wmy} \end{bmatrix}$ with $S_{w} = \sum_{s} w_{s,t}$, $S_{wm} = \sum_{s} w_{s,t}\, g_{\mathrm{mkt},s}$, $S_{wmm} = \sum_{s} w_{s,t}\, g_{\mathrm{mkt},s}^{2}$, $S_{wy} = \sum_{s} w_{s,t}\, y_{s}$, and $S_{wmy} = \sum_{s} w_{s,t}\, g_{\mathrm{mkt},s}\, y_{s}$, Cramer's rule produces:
$$
\hat{\beta}_{i,t} = \frac{S_{w}\, S_{wmy} - S_{wm}\, S_{wy}}{S_{w}\, S_{wmm} - S_{wm}^{2}}, \qquad \hat{\alpha}_{i,t} = \frac{S_{wy} - \hat{\beta}_{i,t}\, S_{wm}}{S_{w}}
$$
which are the textbook OLS slope and intercept formulas with weighted moments substituted for the unweighted ones. EWLS is OLS on a re-weighted sample whose weights age exponentially in time.

___


## Residual Standard Deviation From the Same Machinery

The residual standard deviation falls out of the same sufficient statistics. Expanding the weighted residual sum of squares at the optimum gives:
$$
\sum_{s=1}^{t} w_{s,t}\,(y_{s} - \mathbf{x}_{s}^{\top}\hat{\boldsymbol{\theta}}_{i,t})^{2} = c_{t} - 2\,\hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t} + \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{A}_{t}\,\hat{\boldsymbol{\theta}}_{i,t}
$$
The cross term cancels because $\hat{\boldsymbol{\theta}}_{i,t}$ satisfies the normal equations $\mathbf{A}_{t}\,\hat{\boldsymbol{\theta}}_{i,t} = \mathbf{b}_{t}$, so $\hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{A}_{t}\,\hat{\boldsymbol{\theta}}_{i,t} = \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t}$, and the weighted residual sum of squares collapses to:
$$
\sum_{s=1}^{t} w_{s,t}\,(y_{s} - \mathbf{x}_{s}^{\top}\hat{\boldsymbol{\theta}}_{i,t})^{2} = c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\mathbf{b}_{t}
$$

Dividing by the effective sample weight $S_{w,t} = [\mathbf{A}_{t}]_{1,1}$ gives the residual standard deviation estimate:
$$
\boxed{\;\hat{\sigma}_{\varepsilon,i,t} = \sqrt{\frac{c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\,\mathbf{b}_{t}}{[\mathbf{A}_{t}]_{1,1}}}\quad\blacksquare\;}
$$

This is the exponentially weighted analogue of the OLS variance estimate without the degrees-of-freedom correction; for daily rebalancing with $h \in [21, 126]$ trading days the difference is negligible, and the formula recovers the closed-form estimate from the same three running moments at no extra cost.

___


## Prior Seeding and the Half-Life Knob

Two parameters control how aggressively EWLS adapts: the prior weight $W_{0}$ that anchors the recursion before any new data arrives, and the half-life $h$ that sets how fast old data fades.

Before any observation is processed, the sufficient statistics are seeded as if $W_{0}$ pseudo-observations consistent with a calibrated prior $(\boldsymbol{\theta}_{i,0}, \sigma_{\varepsilon,i,0})$ had already been seen. With $\mathbf{M} = \mathbb{E}[\mathbf{x}\mathbf{x}^{\top}]$ taken from the calibration sample, the seed is given by:
$$
\mathbf{A}_{0} \gets W_{0} \cdot \mathbf{M}, \qquad \mathbf{b}_{0} \gets \mathbf{A}_{0}\,\boldsymbol{\theta}_{i,0}, \qquad c_{0} \gets \boldsymbol{\theta}_{i,0}^{\top}\mathbf{b}_{0} + W_{0}\,\sigma_{\varepsilon,i,0}^{2}
$$
Solving the normal equations against this seed alone recovers $\hat{\boldsymbol{\theta}}_{i,0} = \boldsymbol{\theta}_{i,0}$ and the variance formula returns $\hat{\sigma}_{\varepsilon,i,0} = \sigma_{\varepsilon,i,0}$, so the recursion launches at the prior and only moves when real data arrives. Large $W_{0}$ keeps the calibrated values dominant longer; small $W_{0}$ lets the first few real observations move the estimates appreciably.

The half-life $h$ controls how quickly the seed (and any earlier real observation) is forgotten. After $k$ trading days the weight on the original seed has decayed to $W_{0}\,\delta^{k}$, so the prior's effective weight halves every $h$ days. A short half-life such as $h = 21$ days (approximately one trading month) puts most of the weight on the last few weeks of data, so the estimates track regime shifts quickly but inherit single-day noise. A long half-life such as $h = 126$ days (six months) averages over a much wider window and produces smooth estimates that lag genuine regime transitions. The default $h = 63$ days (one trading quarter) sits between these extremes and is the value the engine ships with for daily rebalancing.

Together $W_{0}$ and $h$ move the estimator along the speed-stability frontier; everything else in the recursion is mechanical.

___


## Summary

The EWLS recursion compresses the entire weighted history of asset $i$'s SIM regression into three running moments $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$, updated each trading day by a single decayed-plus-rank-one step. The parameter estimate $\hat{\boldsymbol{\theta}}_{i,t}$ comes from a $2 \times 2$ solve, and the residual standard deviation $\hat{\sigma}_{\varepsilon,i,t}$ falls out of the same machinery without extra bookkeeping. The prior weight $W_{0}$ and half-life $h$ are the only two knobs the operator sets; everything else is determined by the data and the recursion.

> __Why this matters:__
>
> The Session 3 engine ingests a fresh market day and must produce updated SIM parameters in time for the next rebalance. EWLS makes that constant-memory, constant-compute, and free of any sliding-window bookkeeping. The same three moments that produce $(\hat{\alpha}_{i}, \hat{\beta}_{i})$ also produce $\hat{\sigma}_{\varepsilon,i}$, which the preference-weight formula needs at the same step.

> __Key Takeaways:__
>
> * __Three running moments are sufficient:__ The weighted normal equations show that $(\mathbf{A}_{t}, \mathbf{b}_{t}, c_{t})$ are the only quantities the EWLS estimator needs from the weighted history. No raw observation is retained, and the recursion runs in constant memory.
> * __The recursion is one decayed-plus-rank-one step:__ Advancing time from $t-1$ to $t$ multiplies every running moment by $\delta = 2^{-1/h}$ and adds the today contribution. The same rule applies to all three moments, and the parameter estimate and residual scale follow from a single $2 \times 2$ solve.
> * __Prior weight and half-life are the only knobs:__ $W_{0}$ sets how strongly the calibrated prior anchors the recursion before real data arrives, and $h$ sets how fast that anchor fades once it does. Together they place the estimator on a speed-stability frontier; all other behavior is mechanical.

___
